# 03 — Verify Pipeline (run every session)

Run this notebook at the start of **every Colab session** after mounting Drive.
It confirms the full pipeline from cache loading to DataLoader batches is intact.

**Expected time: ~2 minutes.**

If all 5 checks pass → proceed to training or exploration.

## Session setup (run these cells every session)

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

In [ ]:
# Cell 2: sys.path + path definitions
import sys, os

REPO_DIR     = '/content/piano_amt'
DRIVE_ROOT   = '/content/drive/MyDrive/piano_amt'
MAESTRO_ROOT = f'{DRIVE_ROOT}/maestro-v3.0.0'
CACHE_DIR    = f'{DRIVE_ROOT}/cache'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"MAESTRO_ROOT = {MAESTRO_ROOT}")
print(f"CACHE_DIR    = {CACHE_DIR}")

In [ ]:
# Cell 3: imports
import torch
import numpy as np
import glob
from pathlib import Path

from src.constants import *
from src.audio import load_audio, wav_to_log_mel, load_audio_as_log_mel
from src.midi import midi_path_to_rolls
from src.dataset import MAESTRODataset, load_from_cache
from src.dataloader import get_dataloader

print("✓ All imports successful")

## Check 1: Constants

Asserts that all critical hyperparameters have their expected values from
Hawthorne 2018a §3 and jongwook/onsets-and-frames.

In [ ]:
assert N_MELS == 229,             f"N_MELS={N_MELS}"
assert SAMPLE_RATE == 16000,      f"SAMPLE_RATE={SAMPLE_RATE}"
assert HOP_LENGTH == 512,         f"HOP_LENGTH={HOP_LENGTH}"
assert abs(FRAMES_PER_SECOND - 31.25) < 1e-6
assert MAX_SEGMENT_FRAMES == 640, f"MAX_SEGMENT_FRAMES={MAX_SEGMENT_FRAMES}"
assert N_KEYS == 88,              f"N_KEYS={N_KEYS}"
assert MIN_MIDI == 21,            f"MIN_MIDI={MIN_MIDI}"
assert MAX_MIDI == 108,           f"MAX_MIDI={MAX_MIDI}"

print(f"N_MELS         = {N_MELS}")
print(f"SAMPLE_RATE    = {SAMPLE_RATE}")
print(f"HOP_LENGTH     = {HOP_LENGTH}")
print(f"FRAMES_PER_SEC = {FRAMES_PER_SECOND}")
print(f"MAX_SEG_FRAMES = {MAX_SEGMENT_FRAMES}")
print(f"N_KEYS         = {N_KEYS}")
print(f"MIDI range     = [{MIN_MIDI}, {MAX_MIDI}]")
print("\n✓ Check 1: Constants OK")

## Check 2: Load one mel from cache

In [ ]:
npz_files = sorted(glob.glob(f"{CACHE_DIR}/*.npz"))
assert npz_files, f"No .npz files found in {CACHE_DIR}. Run 02_build_cache.ipynb first."

data = load_from_cache(npz_files[0])

mel = data['mel']  # (229, T)
print(f"File       : {Path(npz_files[0]).name}")
print(f"mel.shape  : {tuple(mel.shape)}")
print(f"mel.dtype  : {mel.dtype}")
print(f"mel.min()  : {mel.min().item():.4f}")
print(f"mel.max()  : {mel.max().item():.4f}")
print(f"mel.mean() : {mel.mean().item():.4f}")

assert mel.shape[0] == N_MELS,  f"mel.shape[0]={mel.shape[0]}"
assert mel.shape[1] > 0,        "mel has zero frames"
assert data['onset'].shape[1] == N_KEYS
assert data['frame'].shape == data['onset'].shape

print("\n✓ Check 2: Cache loading OK")

## Check 3: MAESTRODataset __getitem__

In [ ]:
ds = MAESTRODataset(
    maestro_root=MAESTRO_ROOT,
    split="train",
    cache_dir=CACHE_DIR,
    segment=True,
    max_files=5,
)
print(f"Dataset size: {len(ds)} files")

item = ds[0]
print("\nItem contents:")
for k, v in item.items():
    if hasattr(v, 'shape'):
        print(f"  {k:12s}: {tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:12s}: {v}")

assert item['mel'].shape    == (N_MELS, MAX_SEGMENT_FRAMES)
assert item['onset'].shape  == (MAX_SEGMENT_FRAMES, N_KEYS)
assert item['frame'].shape  == (MAX_SEGMENT_FRAMES, N_KEYS)
assert item['offset'].shape == (MAX_SEGMENT_FRAMES, N_KEYS)
assert item['velocity'].shape == (MAX_SEGMENT_FRAMES, N_KEYS)
assert isinstance(item['audio_path'], str)

print("\n✓ Check 3: Dataset __getitem__ OK")

## Check 4: DataLoader batch

In [ ]:
loader = get_dataloader(
    maestro_root=MAESTRO_ROOT,
    split="train",
    batch_size=2,
    num_workers=0,
    cache_dir=CACHE_DIR,
    max_files=5,
    use_augmentation=False,
    pin_memory=False,
)

batch = next(iter(loader))
B = batch['mel'].shape[0]
print(f"Batch size: {B}")
print("\nBatch contents:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:12s}: {tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:12s}: {v}")

assert batch['mel'].shape    == (B, N_MELS, MAX_SEGMENT_FRAMES)
assert batch['onset'].shape  == (B, MAX_SEGMENT_FRAMES, N_KEYS)
assert batch['frame'].shape  == (B, MAX_SEGMENT_FRAMES, N_KEYS)
assert batch['offset'].shape == (B, MAX_SEGMENT_FRAMES, N_KEYS)

print("\n✓ Check 4: DataLoader batch shapes OK")

## Check 5: Run full verify script

In [ ]:
!python /content/piano_amt/scripts/verify_pipeline.py \
    --maestro_root {MAESTRO_ROOT} \
    --max_files 3

## All checks passed ✓

If all 5 checks above printed `✓ OK` and the script ended with
`ALL CHECKS PASSED ✓`, the pipeline is fully operational for this session.

**What to do next:**
- **Explore the data**: Open `04_explore_data.ipynb`
- **Start training**: `python train.py --maestro_root {MAESTRO_ROOT} --cache_dir {CACHE_DIR}`
- **Add your model**: Replace `_DummyModel` in `train.py` with your
  `OnsetsAndFrames` implementation and re-run.